In [1]:
import os
import cv2
import numpy as np

import tensorflow as tf
from tensorflow import keras

from deepface import DeepFace

DATA_DIRECTORY = 'data'
SUPPORTED_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff')


def load_dataset(base_path, splits):
    all_data = {}
    print(f"--- Loading Dataset from '{base_path}' ---")

    for split in splits:
        split_path = os.path.join(base_path, split)
        if not os.path.isdir(split_path):
            print(f"Warning: Split directory not found, skipping: '{split_path}'")
            continue

        images_with_labels = []
        emotion_folders = [d for d in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, d))]
        if not emotion_folders:
            continue

        for emotion_label in emotion_folders:
            emotion_path = os.path.join(split_path, emotion_label)
            for filename in os.listdir(emotion_path):
                if filename.lower().endswith(SUPPORTED_EXTENSIONS):
                    file_path = os.path.join(emotion_path, filename)
                    img = cv2.imread(file_path)
                    if img is not None:
                        images_with_labels.append((file_path, img, emotion_label))
        all_data[split] = images_with_labels

    return all_data


def preprocess_image_for_model(image_data):
    return cv2.cvtColor(image_data, cv2.COLOR_BGR2RGB)


def predict_emotion(image_data):
    try:
        analysis_result = DeepFace.analyze(
            img_path=image_data,
            actions=['emotion'],
            enforce_detection=False
        )
        if isinstance(analysis_result, list):
            analysis_result = analysis_result[0]

        return analysis_result.get("dominant_emotion", "no_face_detected")
    except Exception:
        return "error"


def main():
    data_subdirectories = ['train', 'test']
    all_loaded_data = load_dataset(DATA_DIRECTORY, data_subdirectories)

    if not all_loaded_data or not any(all_loaded_data.values()):
        print("\nNo data loaded. Please ensure 'data/train' and 'data/test' exist with emotion folders.")
        return

    total_correct = 0
    total_images = 0

    print("\n--- Evaluating Dataset ---")
    for data_split, image_files in all_loaded_data.items():
        if not image_files:
            continue

        correct_predictions = 0
        for _, img_data, true_label in image_files:
            preprocessed_img = preprocess_image_for_model(img_data)
            predicted_label = predict_emotion(preprocessed_img)
            if predicted_label.lower() == true_label.lower():
                correct_predictions += 1

        split_total = len(image_files)
        split_accuracy = (correct_predictions / split_total) * 100 if split_total > 0 else 0
        print(f"{data_split.upper()} Accuracy: {split_accuracy:.2f}% ({correct_predictions}/{split_total})")

        total_correct += correct_predictions
        total_images += split_total

    # Overall accuracy
    if total_images > 0:
        overall_accuracy = (total_correct / total_images) * 100
        print(f"\nFinal Overall Accuracy: {overall_accuracy:.2f}% ({total_correct}/{total_images})")
    else:
        print("\nNo images evaluated.")

    print("\n--- Processing Complete ---")


if __name__ == "__main__":
    main()


--- Loading Dataset from 'data' ---

--- Evaluating Dataset ---


Action: emotion:   0%|          | 0/1 [00:00<?, ?it/s]


KeyboardInterrupt: 